# Install Package

In [386]:
%%capture
!pip install sentence_transformers
!pip install pyspellchecker
!pip install keybert

# Import Modules

In [387]:
%%capture
import re
import pandas as pd
import copy
import spacy
import datetime
from keybert import KeyBERT
import math
import timeit
import numpy as np
from datetime import datetime, timedelta
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Chat Segments

In [388]:
# Chat Segments - Main Idea
start_time = time.perf_counter()
model = SentenceTransformer("all-miniLM-L6-V2")
def  eucledian_distance(A, B):
    return math.dist(A, B)
chat_messages = []
chat_messages_db = pd.read_csv("conversation.csv").head(500)
chat_messages = (
    pd.read_csv("conversation.csv").head(500)['Text']
      .fillna("")
      .astype(str)
      .tolist()
)
def cosine_distance(A, B):
    return 1-cosine_similarity(A, B)[0][0]
def find_last_point(start, midle, finish):
    distance = []
    index = []
    for i in range(start+1, finish+1): 
        Text1 = " ".join(chat_messages[start:i])
        Text2 = " ".join(chat_messages[i:i+1])
        Text3 = " ".join(chat_messages[i:finish+1])
        vec1 = []
        vec2 = []
        vec3 = []
        vec1 = model.encode(Text1)
        vec2 = model.encode(Text2)
        vec3 = model.encode(Text3)
        dist1 = cosine_distance(vec1.reshape(1, -1), vec2.reshape(1, -1))
        dist2 = cosine_distance(vec1.reshape(1, -1), vec3.reshape(1, -1))
        distance.append([dist1, dist2])
        index.append(i)
    ans = -1
    for i in range(1, len(distance)-1):
        if index[i] <= midle:
            continue
        if distance[i][0] > distance[i-1][0] and distance[i][1] > distance[i+1][1] and distance[i][0] > distance[i+1][0] and distance[i][1] > distance[i-1][1]:
            ans = index[i]
    return ans
vectors = []
Text = ""
prev_dist = 0 
prev_rate = 10000
cur_length = 0
Answer = []
i = 0
while i < len(chat_messages):
    vectors.append(model.encode(chat_messages[i]))
    cur_length = cur_length + 1
    Text = Text + "\n" + chat_messages[i]
    cur_vec = model.encode(Text)
    dist = 0
    for j in range(len(vectors)):
        dist = dist + cosine_distance(cur_vec.reshape(1, -1), vectors[j].reshape(1, -1))
    dist = dist / len(vectors)
    if cur_length > 1:
        cur_rate = (dist-prev_dist)/dist
        if cur_rate > prev_rate:
            # Reverse Method
            k = i-1
            reverse_vectors = []
            reverse_text = ""
            reverse_prev_dist = 0
            reverse_prev_rate = 10000
            reverse_cur_length = 0
            reverse_cur_point = -1
            while k >= i - (cur_length-1):
                reverse_cur_length = reverse_cur_length + 1
                reverse_vectors.append(model.encode(chat_messages[k]))
                reverse_text = reverse_text + "\n" + chat_messages[k]
                reverse_cur_vec = model.encode(reverse_text)
                reverse_dist = 0
                for l in range(len(reverse_vectors)):
                    reverse_dist = reverse_dist + cosine_distance(reverse_cur_vec.reshape(1, -1), reverse_vectors[l].reshape(1, -1))
                reverse_dist  = reverse_dist / len(reverse_vectors)
                if reverse_cur_length > 1 :
                    reverse_cur_rate = (reverse_dist-reverse_prev_dist)/reverse_dist
                    if reverse_cur_rate > reverse_prev_rate:
                        reverse_cur_point = k
                    reverse_prev_rate = reverse_cur_rate
                reverse_prev_dist = reverse_dist
                k = k - 1
            easy = -1
            if reverse_cur_point!=-1:
                easy = find_last_point(i-(cur_length-1), reverse_cur_point, i)
            if easy != -1:
                k = easy
                i = k
                Answer.append(k)
                vectors.clear()
                Text = ""
                prev_dist = 0
                prev_rate = 10000
                cur_length = 0
                cur_rate = 10000
                dist = 10000
        prev_rate = cur_rate
    prev_dist = dist
    i = i + 1
Answer.append(len(chat_messages)-1)
Test_answer = []
Text = ""
point = 0
cur_segment = 0
for i in range(len(chat_messages_db)):
    Test_answer.append(cur_segment)
    if Answer[point] == i:
        cur_segment = cur_segment + 1
        point = point + 1
chat_messages_db['Segment'] = Test_answer
        
chat_messages_db.to_csv("conversation_segment_index.csv", index=False)

end_time = time.perf_counter()
print(f"Wall time: {end_time - start_time} seconds")

Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 5290.46it/s]


Wall time: 126.7727134579909 seconds


In [390]:
segments = ["" for _ in range(cur_segment)]
for i in range(len(chat_messages)):
    seg_id = chat_messages_db.iloc[i]["Segment"]
    text = chat_messages_db.iloc[i]["Text"]

    segments[seg_id] += " " + str(text)

In [407]:
segment_embeddings = []
keybert_segments = []
kw_model = KeyBERT(model)
for i in range(len(segments)):
    text = segments[i]
    keywords = kw_model.extract_keywords(
        text,
        use_mmr= True,
        diversity=0.25,
        stop_words="english"
    )
    sentence = " ".join([kw for kw, score in keywords])
    keybert_segments.append(sentence)
    segment_embeddings.append(model.encode(sentence))

In [470]:
import numpy as np
from sklearn.cluster import HDBSCAN

X = np.array(segment_embeddings)
clusterer = HDBSCAN(
    min_cluster_size=3,
    min_samples=1,
    metric="cosine"
)
labels = clusterer.fit_predict(X)
confidences = clusterer.probabilities_
unique_labels = [lbl for lbl in sorted(set(labels)) if lbl != -1]

cluster_data = []

for lbl in unique_labels:
    current_cluster_content = []
    
    # 1. Get the indices of all points in this cluster
    indices = np.where(labels == lbl)[0]
    
    # 2. Calculate the mean embedding for these points
    # This is the "Semantic Average" of the cluster
    cluster_mean_vector = np.mean(X[indices], axis=0)
    
    for i in indices:
        current_cluster_content.append({
            "text": keybert_segments[i],
            #"confidence": confidences[i]
        })
    
    cluster_data.append({
        "cluster_id": lbl,
        "mean_embedding": cluster_mean_vector, # Saved as a numpy array
        "content": current_cluster_content
    })

In [471]:
Topics = []
for each in cluster_data:
    Topics.append(each['mean_embedding'])

In [501]:
def find_probability(Text, Topics):
    T = 1/len(Topics)
    pro_list = [0 for _ in range(len(Topics))]
    for t in range(len(Topics)):
        pro_list[t] = cosine_similarity(model.encode(Text).reshape(1,-1),Topics[t].reshape(1, -1))[0][0]
    # Softmax Function
    total_sum = 0
    pro_list = pro_list - max(pro_list)
    for t in range(len(Topics)):
        total_sum = total_sum + math.exp(pro_list[t]/T)
    for t in range(len(Topics)):
        pro_list[t] = (math.exp(pro_list[t]/T) / total_sum)
    return pro_list

In [502]:
all = [[] for t in range(len(Topics))]
for each_segment in segments:
    pro = find_probability(each_segment, Topics)
    for t in range(len(Topics)):
        all[t].append(pro[t])
        print(t, pro[t])

0 0.0023190444
1 0.020695627
2 0.008950681
3 0.09047007
4 0.0098299645
5 0.005823495
6 0.8243746
7 0.012563667
8 0.024972858
0 0.001824819
1 0.024442554
2 0.0037790074
3 0.04808083
4 0.0054018535
5 0.06650988
6 0.8273775
7 0.015058943
8 0.007524601
0 0.017287878
1 0.029462924
2 0.0278404
3 0.15652739
4 0.02261985
5 0.28238302
6 0.39926723
7 0.04903407
8 0.015577256
0 0.0022659292
1 0.01874867
2 0.0038896499
3 0.009279489
4 0.002413392
5 0.92707545
6 0.02101704
7 0.012486768
8 0.002823634
0 0.0026746495
1 0.016749484
2 0.003263367
3 0.009727797
4 0.004968591
5 0.90232587
6 0.008939137
7 0.044887982
8 0.0064631067
0 0.0019758495
1 0.012711284
2 0.0059736883
3 0.009064873
4 0.003263975
5 0.9186968
6 0.01107987
7 0.025966369
8 0.011267267
0 0.033507004
1 0.056330536
2 0.021134533
3 0.053354274
4 0.1621901
5 0.11736587
6 0.032273278
7 0.15337078
8 0.37047362
0 0.019140387
1 0.008185358
2 0.005227076
3 0.02644515
4 0.8484613
5 0.003383363
6 0.014225512
7 0.04471393
8 0.030217962
0 0.02086224

In [510]:
import numpy as np

topic_thresholds = []

print(f"{'Topic ID':<10} | {'Mean':<8} | {'Std':<8} | {'Knee k':<10} | {'Threshold':<10}")
print("-" * 65)

for t in range(len(Topics)):
    topic_scores = np.array(all[t])
    mu = np.mean(topic_scores)
    sigma = np.std(topic_scores)
    
    k_range = np.arange(0, 1.001, 0.001)
    counts = []
    
    for k in k_range:
        current_threshold = mu + k * sigma
        cnt = np.sum(topic_scores >= current_threshold)
        counts.append(cnt)
    
    counts = np.array(counts)
    
    # --- MANUAL KNEEDLE LOGIC ---
    # 1. Normalize both axes to [0, 1]
    # x is our k_range, y is our counts
    x_norm = (k_range - k_range.min()) / (k_range.max() - k_range.min())
    y_norm = (counts - counts.min()) / (counts.max() - counts.min())
    
    # 2. Define the diagonal line (from start to end of normalized curve)
    # Since it's a decreasing curve, the line goes from (0,1) to (1,0)
    # The distance from a point (px, py) to the line y = -x + 1 is:
    # distance = abs(px + py - 1) / sqrt(2)
    distances = np.abs(x_norm + y_norm - 1) / np.sqrt(2)
    
    # 3. The Knee is the point with the maximum distance
    optimal_k_idx = np.argmax(distances)
    optimal_k = k_range[optimal_k_idx]
    
    # --- FALLBACK ---
    # If the max distance is 0 (flat line), default to a safe value
    if np.max(distances) == 0:
        optimal_k = 0.5
        
    final_threshold = mu + optimal_k * sigma
    topic_thresholds.append(final_threshold)
    
    print(f"{t:<10} | {mu:.4f} | {sigma:.4f} | {optimal_k:.3f} | {final_threshold:.4f}")

Topic ID   | Mean     | Std      | Knee k     | Threshold 
-----------------------------------------------------------------
0          | 0.1228 | 0.2529 | 0.684 | 0.2957
1          | 0.0748 | 0.1736 | 0.360 | 0.1373
2          | 0.0835 | 0.2278 | 0.226 | 0.1350
3          | 0.1499 | 0.2077 | 0.226 | 0.1968
4          | 0.0758 | 0.1681 | 0.266 | 0.1205
5          | 0.0785 | 0.1901 | 0.515 | 0.1764
6          | 0.1335 | 0.2110 | 0.253 | 0.1869
7          | 0.1162 | 0.1908 | 0.102 | 0.1357
8          | 0.1650 | 0.2801 | 0.326 | 0.2563
